In [49]:
from pathlib import Path
import sys
sys.path.append( str( Path("../../../.." ).resolve()) )

# Biopython

In [50]:
# Load test datasets
import gemmi
from Bio.PDB import PDBParser, MMCIFParser, PDBIO, MMCIFIO
from io import StringIO

from xaidar.data.molecModels import flatten_pdb
from xaidar.data.molecModels import get_pdb_stats, get_res_CoM
from xaidar.data.molecModels import  sele_pdb, sele_Lig, sele_AA
ev2a_pdb = gemmi.read_pdb("../../../../tests/testdata/protein/ev2a.pdb")
testmmcif = gemmi.read_structure("../../../../tests/testdata/protein/testprot.mmcif")

lig_pdb = sele_pdb( ev2a_pdb, sele_Lig )
lig_com = get_res_CoM( flatten_pdb( lig_pdb, "residue") )
lig_com = gemmi.Position( *lig_com[0] )

ev2a_pdb = sele_pdb( ev2a_pdb, sele_AA)
# get_pdb_stats( ev2a_pdb)

In [51]:
def gemmi_to_biopy( gemmi_struct: gemmi.Structure, file_type: str = "mmcif"):
    """
    Convert a Gemmi Structure object to a Biopython Structure object.
    Args:
    - gemmi_struct (gemmi.Structure): The Gemmi Structure object to convert.
    - file_type (str): The file format to use for conversion ("pdb" or "mmcif").
    Returns:
    - Bio.PDB.Structure.Structure: The converted Biopython Structure object.
    """
    if file_type == "pdb":
        prot_block = gemmi_struct.make_pdb_string()
        parser = PDBParser(QUIET=True)                                          # Create a PDBParser    
    elif file_type == "mmcif" :
        prot_block =  gemmi_struct.make_mmcif_block().as_string()
        parser = MMCIFParser(QUIET=True)                                        # Create an MMCIFParser
    
    biopython_struct = parser.get_structure(gemmi_struct.name,                  # gemmi_struct.name = ID you assign to the Biopython structure
                                            StringIO(prot_block))               # Use StringIO to treat the string as a file
    return biopython_struct

In [52]:
# Test pdb conversion
biopython_struct = gemmi_to_biopy( ev2a_pdb, file_type="pdb" )
# You now have a Biopython Structure object
print(f"Original Gemmi object: {type(ev2a_pdb)}")
print(f"Converted Biopython object: {type(biopython_struct)}")
print(f"Biopython structure ID: {biopython_struct.id}")

Original Gemmi object: <class 'gemmi.Structure'>
Converted Biopython object: <class 'Bio.PDB.Structure.Structure'>
Biopython structure ID: ev2a


In [53]:
# Test mmcif conversion
biopython_struct = gemmi_to_biopy( ev2a_pdb, file_type="mmcif" )
# You now have a Biopython Structure object
print(f"Original Gemmi object: {type(ev2a_pdb)}")
print(f"Converted Biopython object: {type(biopython_struct)}")
print(f"Biopython structure ID: {biopython_struct.id}")

Original Gemmi object: <class 'gemmi.Structure'>
Converted Biopython object: <class 'Bio.PDB.Structure.Structure'>
Biopython structure ID: ev2a


- ## biopyt_to_gemmi function

In [54]:
def biopy_to_gemmi( biopython_struct: 'Bio.PDB.Structure.Structure', 
                       file_type : str = "mmcif" ) -> gemmi.Structure:
    """
    Convert a Biopython Structure object to a Gemmi Structure object.
    Args:
    - biopython_struct (Bio.PDB.Structure.Structure): The Biopython Structure object to convert.
    - file_type (str): The file format to use for conversion ("pdb" or "mmcif").
    Returns:
    - gemmi.Structure: The converted Gemmi Structure object.
    """
                                                                                # Write Biopython structure to a PDB string
    if file_type == "pdb":
        file_io = PDBIO()
    elif file_type == "mmcif":
        file_io = MMCIFIO()
    else:
        raise ValueError("Unsupported file type. Use 'pdb' or 'mmcif'.")
    string_io = StringIO()
    file_io.set_structure(biopython_struct)
    file_io.save(string_io)
    file_string = string_io.getvalue()                                              # Get the string value from the buffer
                                                                                # Read the PDB/MMCIF string into a Gemmi Structure
    if file_type == "pdb": gemmi_struct = gemmi.read_pdb_string(file_string)
    elif file_type == "mmcif": 
        gemmi_doc = gemmi.cif.read_string(file_string)                           # First, parse the string into a gemmi.cif.Document
        gemmi_struct = gemmi.make_structure_from_block(gemmi_doc.sole_block())   # Then, create a Gemmi Structure from the sole block
    return gemmi_struct

In [62]:
# Test back conversion to Gemmi (PDB)
biopython_struct = gemmi_to_biopy( ev2a_pdb, file_type="mmcif" )
new_ev2a_pdb = biopy_to_gemmi( biopython_struct, file_type="mmcif" )
get_pdb_stats(ev2a_pdb)
print()
get_pdb_stats(new_ev2a_pdb)

from xaidar.data.molecModels import model_seqAlign

align = model_seqAlign( ev2a_pdb, new_ev2a_pdb ).visualize().map_matching_res()

print( [res.seqid.num for res in align.ref_res_lst])
print( [res.seqid.num for res in align.query_res_lst])
if [res.seqid.num for res in align.ref_res_lst] == [res.seqid.num for res in align.query_res_lst]:
    print("\nSequence alignment successful: Residue numbers match.")



####################
Number of models: 1
Number of chains in 1st Model: 2

Chain ID: A
	Number of Residues: 140
	Unique List of Non-A.A.: set()
	Contains A.A.
Chain ID: B
	Number of Residues: 140
	Unique List of Non-A.A.: set()
	Contains A.A.


####################
Number of models: 1
Number of chains in 1st Model: 2

Chain ID: A
	Number of Residues: 140
	Unique List of Non-A.A.: set()
	Contains A.A.
Chain ID: B
	Number of Residues: 140
	Unique List of Non-A.A.: set()
	Contains A.A.
                |         |         |         |         *         |         |         |         |         +         |         |         |         |
Ref:   SGAIYVGNYRVVNRHLATHNDWANLVWEDSSRDLLVSSTTAQGCDTIARCDCQTGVYYCSSRRKHYPVSFSKPSLIFVEASEYYPARYQSHLMLAVGHSEPGDCGGILRCQHGVVGIVSTGGNGLVGFADVRDLLWLDEE
       ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
Query: SGAIYVGNYRVVNRHLATHNDWANLVWEDSSRDLLVSSTTAQGCDTIARCDCQTGVYYCS

# Align Structures

In [ ]:
from Bio.PDB.cealign import CEAligner
from Bio.PDB import PDBIO

reference_structure = ev2a_pdb
mobile_structure = new_ev2a_pdb



# Instantiate the CEAligner object
# The Combinatorial Extension (CE) algorithm is good for finding the best
# structural alignment, even with low sequence similarity.
ce_aligner = CEAligner()

# Set the reference structure
# All subsequent alignments will be against this structure.
ce_aligner.set_reference(reference_structure)

# Align the mobile structure onto the reference.
# This method modifies the `mobile_structure` object in place,
# transforming the coordinates of all its atoms.
print(f"Aligning '{mobile_structure.id}' onto '{reference_structure.id}'...")
ce_aligner.align(mobile_structure)

# The `rms` attribute of the aligner now holds the calculated RMSD.
print(f"\nAlignment complete.")
print(f"RMSD: {ce_aligner.rms:.2f} Angstroms") # .2f formats to 2 decimal places

biopy_to_gemmi(mobile_structure, file_type="pdb")


# # --- Save the aligned structure ---
# # Now that the coordinates of the `mobile_structure` object have been
# # transformed, we can save it to a new PDB file.
# io = PDBIO()
# io.set_structure(mobile_structure)
# io.save(output_aligned_file)

# print(f"\nSuccessfully saved the aligned mobile structure to: {output_aligned_file}")


In [63]:
ev2a_pdb[0]["A"].whole().check_polymer_type()

PolymerType.PeptideL